# Week 7: Orbital mechanics primer: TLEs and Keplerian elements

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/7/](https://launchdetect.com/academy/week/7/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_What is a TLE? What is inclination? What is an ascending node? This week is the gateway from 'satellite' as abstract object to 'satellite' as a geometric trajectory you can plot, predict, and analyze._


## Why this week matters

TLE (Two-Line Element) is the universal wire format for orbital state. Every amateur tracker, every commercial space-situational-awareness system, every public satellite-tracking site reads TLEs. NORAD's 18th Space Defense Squadron generates them; Space-Track.org and CelesTrak distribute them; you'll see one for every trackable object in orbit.

**Why does it matter that you can read them by hand?** Because someday you'll have a TLE in your hand and no internet, and you need to know if it's the LEO ISS or the GEO Inmarsat. The 6 Keplerian elements are visible in 70 characters of plain text — once you can parse them mentally, you can spot the orbit type instantly.

Also because every Python library that propagates orbits (skyfield, sgp4, satellite.js) takes a TLE as input. Knowing what's in those 70 characters is the foundation for everything in weeks 8-10.


## Learning objectives

By the end of this lab you will be able to:

- Read a TLE and identify each of the 6 Keplerian elements
- Distinguish LEO / MEO / GEO / Molniya / SSO from inclination + altitude
- Explain what a ground track is and how it relates to orbital period
- Understand why GPS orbits at MEO and ISS at LEO


## Setup — and why these dependencies

No dependencies needed for the parsing — just Python's string-slicing. We'll add `skyfield` in Week 8 for actual propagation.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Parse a real ISS TLE by hand, extracting each of the 6 Keplerian elements via string slicing at the documented byte offsets. Then sanity-check: does the inclination match Baikonur's latitude? Does the period match a 90-min LEO orbit?

**Why parse by hand instead of using a library?** Because the library hides the structure. After this lab, you'll never look at a TLE the same way again — you'll be able to glance at line 2 and say 'oh, that's a polar SSO at ~700 km altitude' before any code runs.

**Why the ISS specifically?** It's the most-tracked object in orbit, the TLE is always fresh on CelesTrak, and its orbital parameters (51.6° inclination, ~92 min period, ~400 km altitude) are widely known so you can sanity-check your parse.


In [ ]:
# Week 7: read your first TLE.
tle_lines = [
    "1 25544U 98067A   24130.50145833  .00018539  00000-0  33188-3 0  9994",
    "2 25544  51.6406 348.5395 0006703 117.9568 358.1729 15.50289267449420",
]

# Parse Keplerian elements by hand from line 2
line2 = tle_lines[1]
inclination_deg     = float(line2[8:16])
raan_deg            = float(line2[17:25])
eccentricity        = float("0." + line2[26:33])
arg_perigee_deg     = float(line2[34:42])
mean_anomaly_deg    = float(line2[43:51])
mean_motion_rev_day = float(line2[52:63])

print(f"Inclination:    {inclination_deg:.2f}°")
print(f"RAAN:           {raan_deg:.2f}°")
print(f"Eccentricity:   {eccentricity:.7f}")
print(f"Arg of perigee: {arg_perigee_deg:.2f}°")
print(f"Mean anomaly:   {mean_anomaly_deg:.2f}°")
print(f"Mean motion:    {mean_motion_rev_day:.4f} revs/day")
print(f"Period:         {1440/mean_motion_rev_day:.1f} min")

# TODO: fetch the latest ISS TLE from CelesTrak (`requests.get('https://celestrak.org/NORAD/elements/gp.php?CATNR=25544&FORMAT=TLE')`)
# and parse it the same way. Compare today's mean motion to the value above.


## What just happened — and why it works

TLE line 2 is laid out in fixed-width columns per the NORAD spec. Each Keplerian element occupies a specific byte range. You sliced:

- Bytes 8-16: inclination in degrees
- Bytes 17-25: right ascension of ascending node
- Bytes 26-33: eccentricity (with implicit '0.' prefix — note the trick)
- Bytes 34-42: argument of perigee
- Bytes 43-51: mean anomaly
- Bytes 52-63: mean motion (revolutions per day)

The eccentricity field is a quirk: TLEs save 2 characters by omitting the leading '0.' — a TLE eccentricity of '0006703' means 0.0006703. Forget the prefix and your orbit becomes hyperbolic, which would be exciting for the ISS.

Period from mean motion: `period_minutes = 1440 / mean_motion_revs_per_day`. 1440 = minutes per day. For mean motion 15.5028: period = 1440 / 15.5028 = 92.89 minutes. That checks out — ISS LEO.


## Common gotchas

- **Eccentricity 'implicit decimal' trick** — TLE format saves bytes by dropping the leading `0.`. Always prepend it before float parsing.
- **TLE epochs are in UT1, not UTC.** The difference is sub-second but matters for high-precision propagation. `skyfield` handles this automatically; if you're hand-rolling, look up the IERS leap-second / UT1-UTC offset for the epoch's date.
- **TLEs go stale fast.** A TLE more than 14 days old is unreliable for ISS-like low orbits because atmospheric drag isn't well-modeled long-term. Refresh weekly or more often.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/7/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
